# Police versus Guerillas

We will investigate a genre of games called Police versus Guerrillas. The idea is that an insurgency (the guerrilla force) is striking against the government (the police). The police have two arsenals filled with weapons and amunition, and the guerrillas would like to take either one of them, or both, if possible.

**Units**

The police have $p$ units they can send to defend the arsenals while the guerrillas have $g$ units with which they can attack. Since $p\geq 2g$ implies the **police always win** and $p\leq g$ implies the guerrillas always have a winning strategy, we generally are sovling games where
$$g \leq p < 2g$$
Common scenarios would have us analyzing 5P vs 3G, 7P vs 5G, or 8P vs 7G. We will conduct an emprical approach to solving this game, so the initiailization code below will be helpful.

In [58]:
## Do not change this cell, only execute it. 
## This cell initializes Python so that pandas, numpy and scipy packages are ready to use.

from datascience import *
import random
import pandas
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import scipy as sp
import math
import string
import inflect
import scipy.stats as stats
engine = inflect.engine()

We first initialize the values we need:
- $p$ : number of police units
- $g$ : number of guerrilla units
- strat : an arrays of strategies listed by larger number of the two groups of units
- index : the column or row numbers in the results table

In [47]:
p = 8
g = 7

p_strat = np.arange(math.ceil(p/2),p+1)
print(p_strat)
p_index = np.arange(1,len(p_strat)+1)
print(p_index)

g_strat = np.arange(math.ceil(g/2),g+1)
print(g_strat)
g_index = np.arange(1,len(g_strat)+1)
print(g_index)
engine = inflect.engine()

[4 5 6 7 8]
[1 2 3 4 5]
[4 5 6 7]
[1 2 3 4]


In [48]:
def create_player_dict(name, anchor_val):
    """
    Creates a dictionary containing player info, strategy sets, and index sets.
    """
    # Calculate the strategy set: from ceil(n/2) to n (inclusive)
    strat = np.arange(math.ceil(anchor_val / 2), anchor_val + 1)
    
    # Create an index set starting from 1
    index = np.arange(1, len(strat) + 1)
    
    return {
        'player': name,
        'index': index,
        'strat': strat
    }

# Creating the dictionaries
P = create_player_dict('P', p)
G = create_player_dict('G', g)

# Displaying the results
print(P)
print(G)

{'player': 'P', 'index': array([1, 2, 3, 4, 5]), 'strat': array([4, 5, 6, 7, 8])}
{'player': 'G', 'index': array([1, 2, 3, 4]), 'strat': array([4, 5, 6, 7])}


In [49]:
print(P['strat']) ; P['index']

[4 5 6 7 8]


array([1, 2, 3, 4, 5])

In [50]:
def words(num_array):

    return [engine.number_to_words(n) for n in num_array]

def word(n):
   return engine.number_to_words(n).replace("-", "_")

In [51]:
words(P['strat'])

['four', 'five', 'six', 'seven', 'eight']

In [52]:
word(P['strat'][1])

'five'

In [53]:
def play(p_max, g_max, p_A, g_A):
   A = p_A ; B = p_max - A ;
   res = ()
   for k in range(90):
      a = np.random.choice(make_array( g_A, g_max - g_A ) ) ; b = g_max - a ;
      if ( a > A ) or ( b > B ):
         v = 0
      else:
            v = 1
      res = np.append(res, v)
   temp = np.round( ( sum(res) + 5 ) / 100.0 , 3 )
   t = temp.astype(float)
   return np.round(t, 3)

In [54]:
play(5,4,5,3)

0.050000000000000003

In [55]:
for i in range(len(P['index'])):
    print(P['index'][i])

1
2
3
4
5


In [56]:
tab = Table().with_columns('', words(G['strat']))
for t  in range(len(P['index'])):
    tab = tab.with_columns(word(P['strat'][t]),np.arange(len(g_strat)).astype(np.float64)+t)
tab

,four,five,six,seven,eight
four,0,1,2,3,4
five,1,2,3,4,5
six,2,3,4,5,6
seven,3,4,5,6,7


In [57]:
tab[1][0]

0.0

In [16]:
p , g

(8, 7)

In [17]:
math.ceil(p/2) , math.ceil(g/2)

(4, 4)

In [18]:
p_offset = math.ceil(p/2) -1 ; g_offset = math.ceil(g/2) ;
p_offset , g_offset

(3, 4)

In [19]:
np.round(play(p,g,3,4),2)

0.5

In [20]:
tab[6-p_offset][6-g_offset] = play(p,g,6,6)
tab

,four,five,six,seven,eight
four,0,1,2,3,4
five,1,2,3,4,5
six,2,3,0.6,5,6
seven,3,4,5,6,7


In [21]:
tab.column(1)

array([ 0.,  1.,  2.,  3.])

In [22]:
row_1 = np.array(tab.row(0))
row_1

array(['four', '0.0', '1.0', '2.0', '3.0', '4.0'],
      dtype='<U32')

In [23]:
row_1 = row_1[1:]
row_1

array(['0.0', '1.0', '2.0', '3.0', '4.0'],
      dtype='<U32')

In [59]:
def update_ema(current_ema, outcome, alpha=0.2):
    return (outcome * alpha) + (current_ema * (1 - alpha))